In [ ]:
import os, glob, time, json
import torch
t0 = time.time()
def tlog(m): print(f'[t={time.time()-t0:.0f}s] {m}', flush=True)
MEMBERS = {'f0': ('dimaspashaakrilian/rsna-knee-kfold-c5-f0', 'kfold_c5_f0.pt'), 'f1': ('dimaspashaakrilian/rsna-knee-kfold-c5-f1', 'kfold_c5_f1.pt'), 'f2': ('dimaspashaakrilian/rsna-knee-kfold-c5-f2', 'kfold_c5_f2.pt'), 'f3': ('dimaspashaakrilian/rsna-knee-kfold-c5-f3', 'kfold_c5_f3.pt'), 'v3': ('dimaspashaakrilian/rsna-knee-train-coatnet-v3-rawmean', 'coatnet_rawmean_ft_v3.pt'), 'r2': ('dimaspashaakrilian/rsna-knee-train-round2-c5-r1', 'coatnet_fused_r1_c5.pt')}
SOUPS = {'soup_folds4': ['f0', 'f1', 'f2', 'f3'], 'soup_folds4_v3': ['f0', 'f1', 'f2', 'f3', 'v3'], 'soup_all6': ['f0', 'f1', 'f2', 'f3', 'v3', 'r2'], 'soup_v3_r2': ['v3', 'r2']}
def find(fname):
    hits = []
    for pat in ('/kaggle/input/*/' + fname, '/kaggle/input/*/*/' + fname, '/kaggle/input/*/*/*/' + fname, '/kaggle/input/*/*/*/*/' + fname):
        hits += glob.glob(pat)
    hits = sorted(set(hits))
    assert len(hits) == 1, f'{fname}: ditemukan {hits}'
    return hits[0]
paths = {k: find(v[1]) for k, v in MEMBERS.items()}
for k, p in paths.items(): tlog(f'{k}: {p} ({os.path.getsize(p)/1e6:.0f} MB)')
cache = {}
def load(k):
    ck = torch.load(paths[k], map_location='cpu', weights_only=False)
    return ck
ref = load('f0')
arch, res = ref['arch'], ref['res']
keys = list(ref['model'].keys())
tlog(f'arch={arch} res={res} tensor={len(keys)}')
for name, mem in SOUPS.items():
    acc = {k: None for k in keys}
    n = 0
    for m in mem:
        ck = ref if m == 'f0' else load(m)
        assert ck['arch'] == arch and int(ck['res']) == int(res), f'{m}: arch/res beda'
        sd = ck['model']
        assert list(sd.keys()) == keys, f'{m}: kunci state_dict beda'
        for k in keys:
            t = sd[k]
            if t.is_floating_point():
                acc[k] = t.double().clone() if acc[k] is None else acc[k] + t.double()
            else:
                acc[k] = t.clone() if acc[k] is None else acc[k]    # buffer integer: ambil dari anggota pertama
        n += 1
        del ck
    out = {}
    for k in keys:
        out[k] = (acc[k] / n).to(ref['model'][k].dtype) if ref['model'][k].is_floating_point() else acc[k]
        if out[k].is_floating_point():
            assert bool(torch.isfinite(out[k]).all()), f'nonfinite {k}'
    torch.save({'arch': arch, 'res': res, 'model': out}, f'/kaggle/working/{name}.pt')
    tlog(f'{name}: rata-rata {n} model {mem} -> {name}.pt')
tlog('SOUP SELESAI')
